# Following a Cohort Through Life: Why Death Probabilities Are Not Death Rates

*Demographic Analysis Series · Module 1 (Mortality) · Lesson 1.4a*
*Author: Jiajia Li | Published: November 22, 2026 | Reading time: about 25 minutes*

[Lessons 1.1 through 1.3](/blog/series/demographic) compared populations using death rates, and every rate had person-years in its denominator. Following a cohort is a different exercise. The cohort is a fixed group of lives; it shrinks only through death; and what carries forward from one age to the next is the probability of surviving the interval, not a rate. The two quantities count the same deaths and are routinely confused in applied work.

This lesson defines the death probability $q$, distinguishes it from the death rate $m$, derives the simplest conversion between them, and follows Australia's 1988 schedule as a synthetic cohort from 100,000 births to age 85. Lesson 1.4b will turn the survival curve into the columns of the life table.

By the end of this notebook you will be able to:

- Define a death probability and explain why its denominator differs from a rate's
- Derive the uniform-deaths conversion $q_n=n\,m/(1+\frac{n}{2}m)$ and locate its assumptions
- Follow a cohort from birth and read its survival curve
- Explain why death probabilities compound but do not add
- Quantify the error made by treating $n\,m$ as a probability

## 1. Two Denominators

A cohort is a closed group of people who share a starting point in age and time and are followed until death. Let $N$ be the number alive at exact age $x$, $D$ the number who die before age $x+n$, and

$$q_n(x)=\frac{D}{N}$$

the probability that a survivor at $x$ dies within the interval. It is a conditional probability: it is defined only for those who reached $x$.

The death rate over the same interval is $m(x)=D/E$, where $E$ is the person-years lived in $[x,x+n)$. Both quantities count the same deaths. They differ in the denominator: $N$ counts lives at the start of the interval; $E$ counts years lived, including the partial years of those who die inside it. A person who dies at $x+1$ contributes one year to $E$, not zero, and contributes nothing further to $N$.

The two are related. Suppose the $D$ people who die live $a$ years on average inside the interval, with $0\le a\le n$. The $N-D$ survivors each contribute $n$ years and the dying contribute $aD$, so

$$E=n(N-D)+aD=nN-(n-a)D.$$

Then $m=D/E$ and $q_n=D/N$ give

$$q_n(x)=\frac{n\,m(x)}{1+(n-a)\,m(x)}.$$

Three consequences follow immediately:

- $q_n$ is bounded between 0 and 1; $m$ is not bounded above and carries units of per person-year.
- $q_n$ depends on the interval length $n$; $m$ is an annual rate. A probability over five years is not a rate over five years.
- When $m$ is small, $q_n\approx n\,m$; the difference is second order in $m$, which is why the confusion survives at young ages and bites at old ones.

The code below rebuilds the 1988 tables, computes five-year rates for both sexes combined, and prepares the age grid. The open-ended 85+ group is set aside: every member of a cohort eventually dies, so an open interval needs separate treatment, which Lesson 1.6a provides.

In [ ]:
# ---- 1. Load Tables 1.2/1.3 and build five-year rates ----
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from io import StringIO
from IPython.display import display

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Arial'],
    'font.size': 11,
    'figure.dpi': 125,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.linewidth': 0.8,
    'legend.frameon': False,
    'svg.fonttype': 'none',
    'pdf.fonttype': 42,
})
GREEN, RUST = '#35634e', '#a95b3d'

# Carmichael (2016), Ch. 1: Table 1.2 (rates per 1,000) and Table 1.3 (persons per 1,000,000), 1988.
rates_csv = """
age,m_malaysia,m_australia,f_malaysia,f_australia
0,16.6,9.8,12.8,7.6
1-4,1.7,0.5,1.1,0.4
5-9,0.6,0.2,0.5,0.2
10-14,0.6,0.3,0.4,0.2
15-19,1.0,1.1,0.5,0.4
20-24,1.5,1.6,0.6,0.5
25-29,1.7,1.5,0.9,0.5
30-34,2.0,1.4,1.1,0.6
35-39,2.4,1.5,1.5,0.8
40-44,3.4,2.2,2.3,1.2
45-49,5.5,3.4,3.2,2.1
50-54,9.9,6.0,5.9,3.4
55-59,15.4,10.0,9.9,5.5
60-64,25.5,17.3,17.0,8.7
65-69,38.3,27.2,27.8,13.8
70-74,62.7,45.3,46.6,23.5
75-79,86.2,71.9,69.8,40.7
80-84,144.3,110.7,119.0,71.4
85+,173.4,186.6,142.8,147.7
"""
pop_csv = """
age,m_malaysia,m_australia,f_malaysia,f_australia
0,14254,7597,13464,7245
1-4,56773,30324,53804,28954
5-9,61207,37762,58012,35803
10-14,56002,38712,53613,36758
15-19,52776,43626,50772,41743
20-24,50193,40771,49060,39297
25-29,42748,42914,44839,41982
30-34,35704,40234,38326,39933
35-39,30427,38778,31269,38391
40-44,23858,36248,23075,34537
45-49,20473,27923,19767,26357
50-54,17052,23855,17190,22798
55-59,12615,22708,13604,21924
60-64,10043,21764,10513,22336
65-69,7373,17701,8498,19973
70-74,4608,12857,5246,16200
75-79,3430,8712,3976,12555
80-84,1399,4471,1675,7827
85+,963,2319,1399,6111
"""
rates = pd.read_csv(StringIO(rates_csv), index_col='age')
pop = pd.read_csv(StringIO(pop_csv), index_col='age')
deaths = pop * rates / 1000

exposure = pd.DataFrame({
    'Malaysia': pop['m_malaysia'] + pop['f_malaysia'],
    'Australia': pop['m_australia'] + pop['f_australia'],
})
asdr = pd.DataFrame({
    'Malaysia': (deaths['m_malaysia'] + deaths['f_malaysia']) / exposure['Malaysia'] * 1000,
    'Australia': (deaths['m_australia'] + deaths['f_australia']) / exposure['Australia'] * 1000,
})
total_deaths = pd.Series({
    'Malaysia': deaths[['m_malaysia', 'f_malaysia']].sum().sum(),
    'Australia': deaths[['m_australia', 'f_australia']].sum().sum(),
})
cdr = total_deaths / exposure.sum() * 1000

display(pd.DataFrame({'Total deaths (per million pop.)': total_deaths.round(0),
                      'CDR per 1,000': cdr.round(2)}))
assert np.allclose(cdr.to_numpy(), [4.952, 7.251], atol=1e-3)

# Five-year grid. Deaths are aggregated before rates are recomputed because
# the printed 0 and 1-4 rows cannot be averaged: exposure differs.
five_yr = [f'{x}-{x+4}' for x in range(0, 85, 5)] + ['85+']


def to_five_year(values):
    """Aggregate ages 0 and 1-4 into 0-4, then select five-year groups."""
    out = values.copy()
    out.loc['0-4'] = out.loc['0'] + out.loc['1-4']
    return out.loc[five_yr]


deaths5 = pd.DataFrame({
    'Malaysia': to_five_year(deaths['m_malaysia'] + deaths['f_malaysia']),
    'Australia': to_five_year(deaths['m_australia'] + deaths['f_australia']),
})
exposure5 = pd.DataFrame({
    'Malaysia': to_five_year(exposure['Malaysia']),
    'Australia': to_five_year(exposure['Australia']),
})
m5 = deaths5 / exposure5 * 1000
ages_closed = [a for a in five_yr if a != '85+']
display(m5.loc[ages_closed].round(2).rename_axis('Age group'))
assert np.isclose(m5.loc['80-84', 'Australia'], 85.69, atol=0.01)
assert np.isclose(m5.loc['0-4', 'Australia'], 2.11, atol=0.01)

## 2. The Simplest Conversion

The formula in Section 1 becomes operational once we choose $a$. The simplest choice is a uniform distribution of deaths inside each interval, $a=n/2$, which gives

$$q_n(x)=\frac{n\,m(x)}{1+\frac{n}{2}\,m(x)}.$$

If rates are printed per 1,000, divide by 1,000 before applying the formula. The function below does that and is unit-tested against a hand calculation.

The assumption is visible in the denominator: the dying are credited with half an interval of exposure, so deaths are spread evenly between $x$ and $x+n$. They are not. Infants who die in the first year of life contribute far less than half a year, and old people who die in a five-year interval contribute more than half, because mortality rises steeply within the interval. Lesson 1.5a replaces $n/2$ with age-specific separation factors and treats the infant case; the constant-force alternative $q_n=1-e^{-n\,m}$ is developed there too. For this lesson the uniform assumption is enough: it isolates the difference between a rate and a probability from the separate question of within-interval timing.

The table compares $n\,m$ with the converted $q_n$ at selected ages. The two agree to three decimal places at young ages and separate at old ones. At ages 80–84, Australia's five-year rate of 85.7 per 1,000 implies $n\,m=0.428$, but the probability of dying within the interval is 0.353, about 18% lower.

In [ ]:
# ---- 2. Convert rates to probabilities ----
def q_from_rate(m_per_1000, n=5, a=None):
    """Death probability for an n-year interval from an annual death rate.

    m_per_1000: annual death rate per 1,000 person-years.
    n: interval length in years.
    a: average years lived inside the interval by those who die.
       Default n/2, the uniform-deaths assumption.
    """
    m = m_per_1000 / 1000
    a = n / 2 if a is None else a
    return n * m / (1 + (n - a) * m)


# Unit test from the derivation: 1,000 survivors, 100 deaths spread uniformly
# over five years -> 4,750 person-years -> m = 21.0526 per 1,000 and q = 0.1.
assert np.isclose(q_from_rate(100 / 4750 * 1000), 0.1, atol=1e-9)

m_aus = m5['Australia'].loc[ages_closed]
q5 = q_from_rate(m_aus)
q5_exact = 1 - np.exp(-5 * m_aus / 1000)   # constant-force benchmark, developed in 1.5a

selected = ['0-4', '45-49', '60-64', '70-74', '75-79', '80-84']
table = pd.DataFrame({
    'm (per 1,000)': m_aus,
    'n × m': 5 * m_aus / 1000,
    'q (uniform deaths)': q5,
    'q / (n × m)': q5 / (5 * m_aus / 1000),
})
display(table.loc[selected].round(4))
assert (q5 < 5 * m_aus / 1000).all()
assert np.isclose(q5.loc['0-4'], 0.0105, atol=1e-3)
assert np.isclose(q5.loc['80-84'], 0.3529, atol=1e-3)

# Figure 1: n m and q across the age range.
fig, ax = plt.subplots(figsize=(9.6, 4.0), layout='constrained')
x = np.arange(len(m_aus))
ax.plot(x, 5 * m_aus.values / 1000, color=RUST, linestyle='--', marker='s',
        markersize=4, linewidth=1.5, label='n × m, reading the rate as a probability')
ax.plot(x, q5.values, color=GREEN, marker='o', markersize=4, linewidth=1.8,
        label='n q, converted probability')
ax.set_xticks(x)
ax.set_xticklabels(m_aus.index, rotation=45, ha='right')
ax.set_xlabel('Age group')
ax.set_ylabel('Probability of dying within five years')
ax.set_title('A five-year rate is not a five-year probability', loc='left', pad=14)
ax.legend(loc='upper left', fontsize=9)
ax.set_axisbelow(True)
ax.grid(axis='y', color='#e6e8e3', linewidth=0.8)
ax.annotate('0.43', (16, 5 * m_aus.iloc[-1] / 1000), textcoords='offset points',
            xytext=(10, 2), ha='left', va='center', fontsize=8.5, color=RUST)
ax.annotate('0.35', (16, q5.iloc[-1]), textcoords='offset points',
            xytext=(10, 2), ha='left', va='center', fontsize=8.5, color=GREEN)
ax.set_xlim(-0.5, 17.6)
plt.show()

**Figure 1 | A five-year rate is not a five-year probability.** Rust: the five-year rate multiplied by five, which is what a rate delivers if it is treated as if it were a probability. Green: the same schedule converted with the uniform-deaths formula. The two agree where mortality is low — at ages 0–49 the gap is under 1% — and separate as mortality rises. At ages 80–84, $n\,m=0.428$ and $q=0.353$, a gap of 18%. The conversion is not cosmetic at the old ages, where most deaths occur.

## 3. Following a Cohort from Birth to 85

With $q_n(x)$ in hand, follow the cohort. Start with a radix of 100,000 births and reduce the survivors at each age by the death probability:

$$l(x+n)=l(x)\left[1-q_n(x)\right].$$

Two features of the recursion matter. It is multiplicative: each interval's survival probability multiplies the previous stock, so small differences accumulate. And the only state variable is the number alive; rates and person-years have disappeared. This is the cohort view — a fixed group shrinking through time.

The code applies the recursion to Australia's 1988 schedule. The result is a **synthetic cohort**, not an observed one: we take the age-specific rates of one calendar year and imagine a cohort that experiences them at every age. A real cohort born in 1988 would be observed for more than eighty years, during which its mortality rates would change and its members would migrate. Lesson 1.4b confronts that difference directly.

In [ ]:
# ---- 3. Follow a synthetic cohort from birth to 85 ----
def follow_cohort(q, n=5, radix=100000.0):
    """Survivors at exact ages 0, n, 2n, ... from a Series of interval death probabilities."""
    l = {0: radix}
    survivors = radix
    for interval, q_value in q.items():
        survivors *= 1 - q_value
        l[int(interval.split('-')[0]) + n] = survivors
    return pd.Series(l)


l = follow_cohort(q5)
q_naive = 5 * m_aus / 1000          # the rate read directly as a probability
l_naive = follow_cohort(q_naive)
l_exact = follow_cohort(q5_exact)   # constant-force conversion, benchmark only

display(l.loc[[0, 5, 20, 45, 65, 85]].round(0).to_frame('Survivors per 100,000'))
print(f"Survivors to 65: {l[65]:,.0f} ({l[65] / 1000:.1f}%); to 85: {l[85]:,.0f} ({l[85] / 1000:.1f}%)")
print(f"Naive n×m: {l_naive[65]:,.0f} to 65 and {l_naive[85]:,.0f} to 85 "
      f"({l_naive[85] / l[85]:.1%} of the converted count)")
print(f"Constant-force conversion: {l_exact[85]:,.0f} at 85 "
      f"({l_exact[85] / l[85] - 1:+.1%} versus uniform deaths)")

# Malaysia's schedule through the same recursion, for context.
q_mal = q_from_rate(m5['Malaysia'].loc[ages_closed])
l_mal = follow_cohort(q_mal)
print(f"Malaysia's 1988 schedule: {l_mal[65]:,.0f} at 65 and {l_mal[85]:,.0f} at 85")

assert np.isclose(l[65], 82909, atol=2)
assert np.isclose(l[85], 31402, atol=5)
assert np.isclose(l_naive[85], 25962, atol=5)
assert np.isclose(l_exact[85], 31681, atol=5)
assert l_naive[85] < l[85]

# Figure 2: the survival curve, and the error of the naive conversion.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.8, 4.2), layout='constrained',
                               gridspec_kw={'width_ratios': [1.5, 1]})
ax1.plot(l.index, l.values / 1000, color=GREEN, linewidth=2.0, marker='o',
         markersize=3, label='Converted from q (uniform deaths)')
ax1.plot(l_naive.index, l_naive.values / 1000, color=RUST, linewidth=1.6,
         linestyle='--', label='n × m read as the probability')
ax1.axvline(65, color='#b9beb4', linewidth=0.9, linestyle=':')
ax1.axvline(85, color='#b9beb4', linewidth=0.9, linestyle=':')
ax1.annotate(f'{l[85] / 1000:.1f}', (85, l[85] / 1000), textcoords='offset points',
             xytext=(10, 4), ha='left', va='bottom', fontsize=8.5, color=GREEN)
ax1.annotate(f'{l_naive[85] / 1000:.1f}', (85, l_naive[85] / 1000), textcoords='offset points',
             xytext=(10, -4), ha='left', va='top', fontsize=8.5, color=RUST)
ax1.set_xlabel('Age')
ax1.set_ylabel('Survivors per 100 births')
ax1.set_title("Following Australia's 1988 schedule as a cohort", loc='left', pad=14)
ax1.set_xlim(0, 90)
ax1.set_ylim(0, 105)
ax1.legend(loc='lower left', fontsize=8.5)
ax1.set_axisbelow(True)
ax1.grid(axis='y', color='#e6e8e3', linewidth=0.8)

retained = l_naive / l * 100
ax2.plot(retained.index, retained.values, color=RUST, linewidth=1.8,
         marker='s', markersize=3)
ax2.axhline(100, color='#7b8078', linewidth=1, linestyle='--')
ax2.set_xlabel('Age')
ax2.set_ylabel('Naive survivors as % of converted')
ax2.set_title('The error grows with age', loc='left', pad=14)
ax2.set_xlim(0, 90)
ax2.set_ylim(75, 102)
ax2.annotate(f'{retained[65]:.1f}%', (65, retained[65]), textcoords='offset points',
             xytext=(-34, -16), fontsize=8.5, color=RUST)
ax2.annotate(f'{retained[85]:.1f}%', (85, retained[85]), textcoords='offset points',
             xytext=(-30, 10), fontsize=8.5, color=RUST)
ax2.set_axisbelow(True)
ax2.grid(axis='y', color='#e6e8e3', linewidth=0.8)
plt.show()

**Figure 2 | Following Australia's 1988 schedule as a cohort.** Left: survivors per 100 births from age 0 to 85, with the cohort reduced by the converted death probabilities (green) and by the naive reading $n\,m$ (rust). The two curves are indistinguishable until the 60s; by 85 the cohort retains 31.4 per 100 births under the converted probabilities and 26.0 under the naive reading. Right: the naive count as a share of the converted count. The naive reading retains 99.7% of the converted survivors at 65 but only 82.7% at 85; before age 50 the difference is negligible. The conversion matters exactly where the deaths are concentrated.

Two benchmarks calibrate the figure. The constant-force conversion, developed in Lesson 1.5a, leaves 31,681 survivors at 85, only 0.9% above the uniform-deaths result; the choice among reasonable conversion formulas is small here. The choice between converting and not converting is larger: 17% of the cohort at age 85. Malaysia's 1988 schedule, run through the same recursion, leaves 73,874 at 65 and 16,396 at 85; the international comparison belongs to Lesson 1.7c.

## 4. Probabilities Compound; They Do Not Add

Because $q_n$ is a probability over an interval, it does not scale linearly with $n$. Under the uniform-deaths formula,

$$q_n=\frac{n\,m}{1+\frac{n}{2}\,m},$$

which is increasing, concave, and bounded above by 1. The linear guess $n\,q_1$ is neither bounded nor correct. At ages 80–84 in the 1988 Australian schedule, the one-year probability is 0.0822. Five times that is 0.411, but the five-year probability is 0.353. Fifteen times the one-year probability is 1.23 — a probability greater than one — while the fifteen-year probability is 0.783.

The table shows the pattern at three ages. The divergence grows with the level of mortality: at ages 60–64 the linear guess is close, and at 80–84 it leaves the probability scale altogether. This is not a quirk of the formula. The probability of dying within $n$ years approaches 1 as $n$ grows, because everyone eventually dies; a linear extrapolation never learns that.

In [ ]:
# ---- 4. Probabilities compound; they do not add ----
rows = []
for age in ['60-64', '75-79', '80-84']:
    m = m_aus[age]
    q1 = q_from_rate(m, n=1)
    q5 = q_from_rate(m, n=5)
    q15 = q_from_rate(m, n=15)
    rows.append({'Age group': age, 'm (per 1,000)': m, 'q(1)': q1,
                 '5 × q(1)': 5 * q1, 'q(5)': q5, '15 × q(1)': 15 * q1, 'q(15)': q15})
display(pd.DataFrame(rows).set_index('Age group').round(4))

q1_80 = q_from_rate(m_aus['80-84'], n=1)
assert 15 * q1_80 > 1
assert q_from_rate(m_aus['80-84'], n=15) < 1

# Figure 3: q as a function of the interval length, against the linear guess.
n_years = np.arange(1, 21)
fig, ax = plt.subplots(figsize=(9.2, 4.0), layout='constrained')
for age, color in [('60-64', GREEN), ('80-84', RUST)]:
    m = m_aus[age] / 1000
    q_n = n_years * m / (1 + (n_years / 2) * m)
    ax.plot(n_years, q_n, color=color, linewidth=1.9, label=f'q(n), annual rate at ages {age}')
m80 = m_aus['80-84'] / 1000
q1_80_float = m80 / (1 + m80 / 2)
ax.plot(n_years, n_years * q1_80_float, color=RUST, linewidth=1.3, linestyle='--',
        alpha=0.8, label='n × q(1), ages 80–84')
ax.axhline(1.0, color='#7b8078', linewidth=1, linestyle=':')
ax.text(1.3, 1.03, 'certainty: q = 1', fontsize=8.5, color='#7b8078')
ax.set_xticks([1, 5, 10, 15, 20])
ax.set_xlabel('Years followed (n)')
ax.set_ylabel('Probability of dying within n years')
ax.set_title('Probabilities compound; they do not add', loc='left', pad=14)
ax.set_xlim(0.5, 20.5)
ax.set_ylim(0, 1.7)
ax.legend(loc='upper left', fontsize=9)
ax.set_axisbelow(True)
ax.grid(axis='y', color='#e6e8e3', linewidth=0.8)
plt.show()

**Figure 3 | Probabilities compound; they do not add.** Solid: the probability of dying within $n$ years, with the annual rate held fixed at the 1988 Australian level of ages 60–64 (green) and 80–84 (rust). Dashed: the linear extrapolation $n\,q_1$ for the age 80–84 rate. The linear line crosses the certainty line $q=1$ near $n=12$ and keeps rising; the converted probability bends toward 1 and never crosses it. At ages 60–64 the curve is nearly linear because the annual probability is small; at 80–84 the curvature is the point.

## 5. What the Cohort View Buys

Three lessons are worth carrying forward.

1. **$q$ is the natural state variable for a cohort; $m$ is the natural measure for a period.** The survival recursion $l(x+n)=l(x)(1-q_n)$ needs probabilities. A period comparison needs rates, because rates are what age-standardization reweights (Lessons 1.2–1.3). The two live in different accounting systems, and converting between them is a modelling step, not a definition.
2. **The conversion depends on assumptions that are invisible in the rate.** The uniform-deaths formula is a placeholder for the average timing of deaths inside the interval. The value of $a$ matters most where mortality is high, and Lesson 1.5a shows how to choose it and what happens to the infant group.
3. **A survival curve is not a prediction.** The synthetic cohort summarizes one period's mortality schedule as an imagined lifetime. It does not say that a child born in 1988 would have faced the 1988 rates at age 80. Lesson 1.4b makes this distinction the centre of the life table.

The survival curve built here is the spine of the life table; the next lesson labels its columns.

## Check Your Understanding

1. A cohort of 1,000 people reaches exact age 80. Their mortality is constant at $m=85.69$ per 1,000 person-years. Using the uniform-deaths conversion, what is the probability that a member dies before age 85, and how many survive to 85?
2. At the same rate, the one-year death probability is 0.0822 and the five-year probability is 0.3529. Explain in one sentence why five times 0.0822 is not the five-year probability.
3. A report gives "the probability of dying in the next five years is 0.12" for a population, with no further detail. What do you need to know before using it in a cohort survival calculation, and what would you do with it?

**Answers.** (1) $q_5=5(0.08569)/[1+2.5(0.08569)]=0.3529$; survivors $=1{,}000(1-0.3529)=647.1$, so about 647. (2) Five-year survival is the product $(1-q_1)^5$, not $1-5q_1$; compounding a probability bends the five-year death probability below the linear multiple, and at high mortality the linear multiple can exceed 1. (3) You need the age interval (the five-year interval starting at which age) and the assurance that it is a probability for that cohort, not a rate. Use $l(x+5)=l(x)(1-0.12)$; if the source reports a rate, convert it with the same interval and an explicit $a$; and check whether the interval is open-ended and the population closed.

## Coming Next: The Life Table's Columns

Lesson 1.4b starts from a historical cohort — the children in King Edward's family — and builds the columns of the life table around it: $l_x$, $d_x$, and $q_x$. It also confronts what the cohort view costs: a true cohort takes a century to observe, which is why demographers construct period tables from a single year's rates.

[Back to the Demographic Analysis Series](/blog/series/demographic) · [Review Lesson 1.3b](/blog/demographic-m1l3b-direct-vs-indirect) · [Review Lesson 1.1a](/blog/demographic-m1l1a-lexis-diagram)

## References and Data Sources

1. Wachter, K. W. (2014). *Essential Demographic Methods*. Harvard University Press. Cohort mortality and survival: Ch. 3, §3.1–3.2. [Publisher's book page](https://www.hup.harvard.edu/books/9780674045576).
2. Carmichael, G. A. (2016). *Fundamentals of Demographic Analysis: Concepts, Measures and Methods*. Springer. Data: Ch. 1, Table 1.2 (age-specific death rates, per 1,000) and Table 1.3 (population by age and sex, per 1,000,000), Malaysia and Australia, 1988. The life table and the rate-to-probability conversion: Ch. 4, pp. 139–145. [Springer book and chapter listing](https://link.springer.com/book/10.1007/978-3-319-23255-3).
3. Preston, S. H., Heuveline, P., & Guillot, M. (2001). *Demography: Measuring and Modeling Population Processes*. Blackwell. The life table and its columns: Ch. 3. [Publisher's book page](https://www.wiley.com/en-us/Demography%3A+Measuring+and+Modeling+Population+Processes-p-9781557864512).

**Reproducibility.** All data are embedded in the code cells, so execution requires no external downloads. Dependencies are NumPy, pandas, Matplotlib, and Jupyter/IPython. The computations are deterministic; no random seed is needed. Assertions check the five-year Australian rates, the unit test of the conversion function, the inequality $q<n\,m$ and the 0–4 and 80–84 values, the survival counts to 65 and 85, the naive and constant-force benchmarks, and the concavity of $q$ in the interval length.